# Supervised fine-tuning (SFT) of the from-scratch model

This notebook continues training a checkpoint produced by `01-pretrain-from-scratch.ipynb`. It uses the repository's own small decoder-only model and byte tokenizer; it does not load arbitrary Hugging Face or Ollama models. Each training line must be JSONL with string `prompt` and `response` fields. Loss is applied only to response tokens. At least two records are required to create a train/validation split.

In [ ]:
import os, sys, json, platform, subprocess
from pathlib import Path
import torch

REPO_URL = "https://github.com/MKarthik730/large-language-models.git"
REPO_DIR = Path("/kaggle/working/large-language-models" if Path("/kaggle/working").exists() else "/content/large-language-models")
if not (REPO_DIR / "training" / "scripts" / "train_sft.py").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Python:", platform.python_version(), "PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))
# PyTorch is normally preinstalled in Kaggle/Colab runtimes. This notebook uses that runtime build.

## Set input paths

`BASE_CHECKPOINT` must point to the pretraining checkpoint. On Kaggle, add it as a notebook input; on Colab, upload it or use mounted Drive. `SFT_FILE` points to your JSONL file. The tiny built-in example is only for testing the pipeline; use a suitable, licensed, representative dataset for real experiments.

In [ ]:
WORK_DIR = Path("/kaggle/working/sft-run" if Path("/kaggle/working").exists() else "/content/sft-run")
WORK_DIR.mkdir(parents=True, exist_ok=True)
BASE_CHECKPOINT = ""  # path to base.pt from the pretraining notebook
SFT_FILE = ""  # path to your .jsonl file
if not BASE_CHECKPOINT:
    raise ValueError("Set BASE_CHECKPOINT to your saved pretraining checkpoint")
if not Path(BASE_CHECKPOINT).is_file(): raise FileNotFoundError(BASE_CHECKPOINT)
if SFT_FILE:
    sft_path = Path(SFT_FILE)
    if not sft_path.is_file(): raise FileNotFoundError(sft_path)
else:
    sft_path = WORK_DIR / "demo-sft.jsonl"
    examples = [
        {"prompt": "What does a language model predict?", "response": "It predicts the next token from prior context."},
        {"prompt": "What is attention?", "response": "Attention combines information from relevant context positions."},
        {"prompt": "What is pretraining?", "response": "Pretraining learns general patterns from a large text corpus."},
        {"prompt": "What is supervised fine-tuning?", "response": "It trains a model on examples of prompts and desired responses."},
        {"prompt": "What is a tokenizer?", "response": "A tokenizer converts text into discrete model input IDs."},
    ]
    sft_path.write_text("\n".join(json.dumps(row) for row in examples) + "\n", encoding="utf-8")

STEPS = 50
BATCH_SIZE = 4
SEQUENCE_LENGTH = 64  # must be <= the base checkpoint context length
OUTPUT_CHECKPOINT = WORK_DIR / "sft.pt"
print("SFT data:", sft_path, "records:", sum(1 for line in sft_path.read_text(encoding="utf-8").splitlines() if line.strip()))

In [ ]:
subprocess.run([
    sys.executable, "-m", "training.scripts.train_sft",
    "--init-checkpoint", str(BASE_CHECKPOINT),
    "--data", str(sft_path),
    "--output", str(OUTPUT_CHECKPOINT),
    "--steps", str(STEPS),
    "--batch-size", str(BATCH_SIZE),
    "--sequence-length", str(SEQUENCE_LENGTH),
    "--device", "cuda" if torch.cuda.is_available() else "cpu",
], cwd=REPO_DIR, check=True)
print("SFT checkpoint:", OUTPUT_CHECKPOINT, "size:", OUTPUT_CHECKPOINT.stat().st_size)

In [ ]:
subprocess.run([
    sys.executable, "-m", "training.scripts.generate",
    "--checkpoint", str(OUTPUT_CHECKPOINT),
    "--prompt", "What does a language model predict?",
    "--max-new-tokens", "100",
    "--device", "cuda" if torch.cuda.is_available() else "cpu",
], cwd=REPO_DIR, check=True)

## Save the result

Copy the SFT checkpoint to persistent storage before the Kaggle/Colab runtime ends. The held-out subset is a small diagnostic only; use a separate evaluation set and inspect model outputs before drawing conclusions.

In [ ]:
try:
    from google.colab import files
except ImportError:
    print("Download or persist manually from", OUTPUT_CHECKPOINT)
else:
    files.download(str(OUTPUT_CHECKPOINT))